[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dragomier/Machine_Learning_2026/blob/main/Homework10.ipynb)

# Exercise 1
We will use the definition of LeNet5 from lab and use the fact that we have also trained it to test how well it will "train" the random photo to resemble some digit. Then we will compare the results with MLP from previous homework.

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()

        self.dropout = nn.Dropout(0.05)
        self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 6,  kernel_size= (5,5), padding = 2)
        self.pool1 = nn.MaxPool2d(kernel_size = (2,2), stride = 2)
        self.conv2 = nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size= (5,5))
        self.pool2 = nn.MaxPool2d(kernel_size = (2,2), stride = 2)
        self.linear1 = nn.Linear(16*5*5, 120)
        self.linear2 = nn.Linear(120, 84)
        self.linear3 = nn.Linear(84, 10)

        self.sigm = torch.nn.Sigmoid()
    def forward(self, x):
        x = self.conv1(x)
        x = self.sigm(x)

        x = self.pool1(x)

        x = self.conv2(x)
        x = self.sigm(x)

        x = self.pool2(x)

        x = torch.flatten(x, 1)

        x = self.linear1(x)
        x = self.sigm(x)

        x = self.linear2(x)
        x = self.sigm(x)

        x = self.dropout(x)
        x = self.linear3(x)

        return x

In [5]:
import torch
from google.colab import drive
drive.mount('/content/drive')

weights_path = "/content/drive/MyDrive/Colab Notebooks/Homework_10/cnn_weights.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

neural_net = LeNet5()
neural_net.load_state_dict(torch.load(weights_path, map_location="cpu"))
neural_net.to(device)
for param in neural_net.parameters():
    param.requires_grad = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
def train_photo(random_noise_photo, targets, lambda_par):
  optimizer = torch.optim.Adam([random_noise_photo], lr=0.05)
  epochs_state = []
  predictions = []
  confidence = []
  epoch = 0
  while epoch < 1000:
      optimizer.zero_grad()
      neural_output = neural_net(random_noise_photo)

      loss = torch.nn.functional.cross_entropy(neural_output, targets, reduction = "mean")
      loss += lambda_par * torch.sum(random_noise_photo ** 2)
      loss.backward()

      optimizer.step()
      if epoch % 10 == 0:
        epochs_state.append(random_noise_photo.clone().detach())
        predictions.append(torch.argmax(neural_output, dim=1))
        probs = torch.softmax(neural_output, dim=1)
        confidence.append(probs[torch.arange(len(targets)), targets].detach().cpu())
      epoch += 1
  return epochs_state, predictions, confidence

In [7]:
from numpy._core.fromnumeric import nonzero
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.pyplot as plt

def make_animation(number, epoch_state, confidence, lambda_par=0, message = None):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f"Training progress for lambda = {lambda_par}" if message is None else message, fontsize=16)

    img = epoch_state[0][number].detach().cpu().squeeze()
    image_plot = ax[0].imshow(img, cmap="gray", animated=True)
    ax[0].set_title(f"Digit {number}")
    ax[0].axis("off")

    ax[1].set_xlim(0, len(epoch_state))
    ax[1].set_ylim(0, 1)
    ax[1].set_title("Confidence")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Prob")

    line, = ax[1].plot([], [], animated=True)

    def update(frame):
        img = epoch_state[frame][number].detach().cpu().squeeze()
        image_plot.set_data(img)
        ax[0].set_title(f"Digit {number}, epoch {frame * 10}")

        conf_values = [confidence[k][number] for k in range(frame + 1)]
        line.set_data(list(range(frame + 1)), conf_values)

        return [image_plot, line]

    anim = FuncAnimation(fig, update, frames=int(len(epoch_state)), interval=200, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [8]:
lambdas = [0, 0.0001, 0.001, 0.01, 0.1, 1, 10]
for lambda_par in lambdas:
  random_noise_photo = torch.randn(size = [10, 1, 28, 28], device=device, requires_grad = True)
  targets = torch.arange(10).to(device)
  epoch_state, predictions, confidence = train_photo(random_noise_photo, targets, lambda_par)
  display(make_animation(0, epoch_state, confidence, lambda_par))

Output hidden; open in https://colab.research.google.com to view.

# Conclusion
Images resemble real MNIST images, especailly for small $\lambda$'s, like 0.0001 or 0.001. This means that CNN generates more realistic pictures than MLP from previous homework. That is because filters in CNN works in the same way as human vision and try to capture characteristic parts of a photo like contrast, edges, texture etc. Then they use those features to distinguish between classes.

# Exercise 2
In this exercise we will try to trick the neural network to believe that the photo of some digit from MNIST dataset, disturbed by some little $\delta$ is really another digit.  

In [9]:
import torchvision
trainset = torchvision.datasets.MNIST(root='./data',
                                      train=True,
                                      download=True,
                                      transform=None)

In [10]:
import matplotlib.pyplot as plt
digits = []
for digit in range(9):
  indices = torch.where(trainset.targets == digit)[0][:10]
  digits.append(trainset.data[indices])


In [11]:
def train_photo_delta(digit, mnist_photos, targets, lambda_par, delta):
  optimizer = torch.optim.Adam([delta], lr=0.1)
  epochs_state = []
  predictions = []
  confidence = []
  good = 10 * [0]
  epoch = 0
  while epoch < 1000:
      optimizer.zero_grad()
      neural_output = neural_net(mnist_photos + delta)

      loss = torch.nn.functional.cross_entropy(neural_output, targets, reduction = "mean")
      loss += lambda_par * delta.pow(2).mean()
      loss.backward()

      optimizer.step()


      if epoch % 10 == 0:
          epochs_state.append((mnist_photos + delta).clone().detach())
          predictions.append(torch.argmax(neural_output, dim=1))
          probs = torch.softmax(neural_output, dim=1)
          confidence.append(probs[torch.arange(len(targets)), targets].detach().cpu())

      for i in range(10):
        good[i] = max(good[i], probs[i, i].item())
      epoch += 1

  return epochs_state, predictions, confidence, good

In [12]:
number_of_animations = 0
for digit in range(9):
  mnist_photos = digits[digit].unsqueeze(1).float().to(device)
  targets = torch.arange(10, device=device)
  lambda_pars = [0.01]

  for lambda_par in lambda_pars:
    delta = (torch.zeros_like(mnist_photos, device = device)).requires_grad_(True).to(device)
    epochs_state, predictions, confidence, good = train_photo_delta(digit, mnist_photos, targets, lambda_par, delta)
    for i in range(9):
      if(good[i] > 0.95 and i != digit):
        display(make_animation(digit, epochs_state, confidence, lambda_par, message = f"Neural network recognises {digit} as {i}"))
        number_of_animations += 1
      if(number_of_animations == 5):
        break
    if(number_of_animations == 5):
      break
  if(number_of_animations == 5):
    break

Output hidden; open in https://colab.research.google.com to view.

# Conclusion
I drawed first 5 animations to be able to upload my solution via GitHub. Pictures are unrecognizable for humans from the original one, although the neural network is tricked to believe that these are not correct digits.